# wisconsin: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('34_wisconsin_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,97.069,95.867,94.725,97.074,97.074,97.074,96.079,98.927,96.468
smote,96.954,95.674,94.365,97.074,96.849,96.962,95.505,98.823,96.314
bsmote,97.089,95.605,93.433,97.908,96.286,97.097,95.021,98.681,96.347
ros,96.840,95.484,94.018,97.074,96.624,96.849,95.652,98.834,96.162
adasyn,97.087,95.611,93.463,97.908,96.286,97.097,95.120,98.711,96.349
casb,96.993,95.594,93.820,97.491,96.510,97.001,94.712,98.668,96.293
casb_relaxed,96.992,95.598,93.838,97.491,96.512,97.001,95.379,98.711,96.295
casb_edited,96.825,95.300,93.243,97.491,96.175,96.833,95.020,98.606,96.062
casb_ros,96.993,95.594,93.820,97.491,96.510,97.001,94.554,98.650,96.293


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,96.446,95.381,95.397,95.399,97.523,96.461,97.367,98.882,95.914
smote,96.431,95.207,94.673,95.820,97.074,96.447,97.372,98.868,95.819
bsmote,96.778,95.357,93.778,97.083,96.510,96.797,96.054,98.607,96.067
ros,96.601,95.500,95.226,95.820,97.411,96.615,97.159,98.839,96.050
adasyn,96.680,95.340,94.136,96.658,96.735,96.696,96.839,98.806,96.010
casb,96.406,94.942,93.586,96.454,96.399,96.427,95.890,98.494,95.674
casb_relaxed,96.580,95.240,94.116,96.445,96.736,96.591,96.413,98.629,95.910
casb_edited,96.513,95.052,93.582,96.658,96.399,96.529,95.630,98.442,95.782
casb_ros,96.360,94.930,93.738,96.241,96.512,96.376,96.214,98.560,95.645


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,96.912,95.751,94.922,96.658,97.186,96.922,98.292,99.203,96.331
smote,96.912,95.751,94.922,96.658,97.186,96.922,98.261,99.196,96.331
bsmote,97.478,96.291,94.801,97.912,97.074,97.493,97.951,99.117,96.885
ros,97.065,95.869,94.758,97.079,97.074,97.076,98.463,99.274,96.467
adasyn,97.422,96.201,94.632,97.912,96.961,97.437,97.845,99.136,96.812
casb,97.423,96.201,94.636,97.917,96.961,97.439,98.323,99.217,96.812
casb_relaxed,97.331,96.183,94.967,97.496,97.186,97.341,98.205,99.166,96.757
casb_edited,97.627,96.324,94.258,98.542,96.736,97.639,97.147,98.973,96.975
casb_ros,97.332,96.185,94.960,97.491,97.186,97.339,98.055,99.138,96.758


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,96.455,95.399,95.416,95.408,97.523,96.465,98.926,99.492,95.927
smote,96.608,95.515,95.245,95.824,97.411,96.618,98.889,99.478,96.061
bsmote,96.955,95.672,94.352,97.074,96.849,96.962,98.906,99.474,96.313
ros,96.551,95.416,95.057,95.824,97.298,96.561,98.926,99.495,95.984
adasyn,97.117,95.876,94.541,97.283,96.961,97.122,98.953,99.495,96.496
casb,96.593,95.341,94.510,96.241,96.961,96.601,98.887,99.473,95.967
casb_relaxed,96.963,95.766,94.748,96.866,97.074,96.970,98.878,99.466,96.364
casb_edited,97.187,95.717,93.512,98.125,96.286,97.205,99.089,99.556,96.452
casb_ros,96.593,95.341,94.510,96.241,96.961,96.601,98.927,99.483,95.967


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,96.720,95.600,95.115,96.135,97.326,96.731,97.666,99.126,96.160
smote,96.726,95.537,94.801,96.344,97.130,96.737,97.507,99.091,96.131
bsmote,97.075,95.731,94.091,97.494,96.680,97.087,96.983,98.970,96.403
ros,96.764,95.567,94.765,96.449,97.102,96.776,97.550,99.111,96.166
adasyn,97.076,95.757,94.193,97.440,96.736,97.088,97.189,99.037,96.417
casb,96.854,95.520,94.138,97.026,96.708,96.867,96.953,98.963,96.187
casb_relaxed,96.966,95.697,94.417,97.074,96.877,96.976,97.219,98.993,96.331
casb_edited,97.038,95.598,93.649,97.704,96.399,97.051,96.721,98.894,96.318
casb_ros,96.819,95.512,94.257,96.866,96.792,96.829,96.938,98.958,96.166


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,96.993,bsmote,97.089,-0.096,LOSS
1,f1,casb_relaxed,95.598,smote,95.674,-0.076,LOSS
2,precision,casb_relaxed,93.838,smote,94.365,-0.526,LOSS
3,recall,casb,97.491,bsmote,97.908,-0.417,LOSS


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,96.580,bsmote,96.778,-0.197,LOSS
1,f1,casb_relaxed,95.240,ros,95.500,-0.260,LOSS
2,precision,casb_relaxed,94.116,ros,95.226,-1.110,LOSS
3,recall,casb_edited,96.658,bsmote,97.083,-0.426,LOSS


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,97.627,bsmote,97.478,0.149,WIN
1,f1,casb_edited,96.324,bsmote,96.291,0.032,WIN
2,precision,casb_relaxed,94.967,smote,94.922,0.044,WIN
3,recall,casb_edited,98.542,bsmote,97.912,0.629,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,97.038,adasyn,97.076,-0.039,LOSS
1,f1,casb_relaxed,95.697,adasyn,95.757,-0.060,LOSS
2,precision,casb_relaxed,94.417,smote,94.801,-0.384,LOSS
3,recall,casb_edited,97.704,bsmote,97.494,0.209,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.973430,0.968421,0.978723,0.958333,0.988764,0.973549,0.995077,0.996957,0.970925,None
1,42,1,svm,smote,0.50,0.973430,0.968421,0.978723,0.958333,0.988764,0.973549,0.995077,0.996957,0.970925,None
2,42,1,svm,bsmote,0.75,0.972708,0.959184,0.940000,0.979167,0.966292,0.972729,0.983140,0.992509,0.965946,None
3,42,1,svm,ros,0.50,0.973430,0.968421,0.978723,0.958333,0.988764,0.973549,0.995077,0.996957,0.970925,None
4,42,1,svm,adasyn,0.75,0.978347,0.969072,0.959184,0.979167,0.977528,0.978347,0.996055,0.997659,0.973710,None
5,42,1,svm,casb,0.50,0.973430,0.968421,0.978723,0.958333,0.988764,0.973549,0.995077,0.996957,0.970925,None
6,42,1,svm,casb_relaxed,0.50,0.973430,0.968421,0.978723,0.958333,0.988764,0.973549,0.995077,0.996957,0.970925,None
7,42,1,svm,casb_edited,0.75,0.972708,0.959184,0.940000,0.979167,0.966292,0.972729,0.996173,0.997659,0.965946,None
8,42,1,svm,casb_ros,0.50,0.973430,0.968421,0.978723,0.958333,0.988764,0.973549,0.995077,0.996957,0.970925,None
9,42,1,knn,original,0.50,0.973430,0.968421,0.978723,0.958333,0.988764,0.973549,0.985589,0.988647,0.970925,None
